---
categories: ["Python", "IR"]
---

# Information Retrieval: Newsgroups

<img src="IR.jpg" width="600">

Information Retrieval methods performed on the classic [20 newsgroups text dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html). Originally completed for DSCI 563 at UBC. 

Methods:
- Whoosh index
- Boolean IR
- Okapi BM25

In [ ]:
import numpy as np
from math import log
from scipy.spatial.distance import pdist,squareform
from collections import defaultdict,Counter
from sklearn.datasets import fetch_20newsgroups
from nltk.stem import PorterStemmer
from nltk import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction import DictVectorizer
from whoosh.qparser import *
from whoosh.fields import Schema, TEXT, KEYWORD, NUMERIC
from whoosh.analysis import RegexTokenizer, LowercaseFilter, StopFilter, StemFilter
from whoosh import index 

### Exercise 1: Whoosh index

In this lab, we will be using a classic IR corpus, the "20 newsgroups" dataset which consists of over 11k posts from 20 newsgroups, [accessible through sklearn](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html). The provided code below extracts the information from the original plaintext format into a list of dictionaries `newsgroup_info_dicts`, where the dictionary for each post includes not just the `Text` body, but also the `Newsgroup`, the original poster (`From`), their organization (`Organization`) and the `Subject` of the post.  Before moving on to the later parts of the lab, take a look at what the post looks like - you'll be indexing specific information in the post, so it is important that you understand how the data looks.

In [ ]:
def get_post_info_dict(full_text):
    '''given a newsgroup post in typical format, extracts the header as a 
    dict, including the body of the text in the "text" field'''
    info_dict = {}
    header_boundary = full_text.find("\n\n")  ### Header is separated by 2 newlines
    for line in full_text[:header_boundary].split("\n"): ### Split the header by lines
        first_colon = line.find(": ") ### Everything before the colon is a topic (ie, author, subject, etc.); everything after is more information
        info_dict[line[:first_colon]] = line[first_colon +2:]
    info_dict["Text"] = full_text[header_boundary + 2:].strip()
    return info_dict

newsgroups = fetch_20newsgroups(remove=('footers', 'quotes'))
newsgroup_info_dicts = []
print("LENGTH: ", len(newsgroups.data))
for i in range(len(newsgroups.data)):
    info_dict =  get_post_info_dict(newsgroups.data[i])
    info_dict["Newsgroup"] = newsgroups.target_names[newsgroups.target[i]]
    if 20 < len(info_dict["Text"]) < 50000: # get rid of very big or very small texts
        newsgroup_info_dicts.append(info_dict)
newsgroup_info_dicts[0]

LENGTH:  11314


{'From': "lerxst@wam.umd.edu (where's my thing)",
 'Subject': 'WHAT car is this!?',
 'Nntp-Posting-Host': 'rac3.wam.umd.edu',
 'Organization': 'University of Maryland, College Park',
 'Lines': '15',
 'Text': 'I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 'Newsgroup': 'rec.autos'}

#### Exercise 1.1 
rubric={accuracy:2}

First, you're going to define two different analyzers:

1. An analyzer for the text and subject fields which tokenizes, lowercases, removes stopwords, and stems, in that order (if you put the stemmer first, it will block some stopword removal!  Think a bit about why this happens.).
2. An analyzer for the newsgroup fields which tokenizes on ".". This will tokenize newsgroups such as "comp.os.ms-windows.misc" into "comp", "os", "ms-windows", "misc". No filters needed for this analyzer.  Hint: The regex tokenizer is looking for a pattern to match in each of its words - not the pattern to split on.  How will you write a regex that captures everything *except* the period?

You will then create a schema that contains the analyzers for the text, subject, author, and organization fields.

You will not be tokenizing the author or organization fields, so you can use the KEYWORD analyzer (This is very similar to how we stored the "genre" type in class - by specifying KEYWORD, you're telling the index not to do any special parsing).  

I recommend that you create a QueryParser for the text, group, author, and organization, and test that they are parsing correctly.

"We are testing the parser" in the text should return (text:test AND text:parser), while "comp.os.ms-windows.misc" in the group should return (group:comp AND group:os AND group:ms-windows AND group:misc).  The author and organization should only split by word.

If you're curious, test queries that have "OR", and "NOT", as well.

In [ ]:
text_analyzer = RegexTokenizer() | LowercaseFilter() | StopFilter() | StemFilter()
newsgroup_analyzer = RegexTokenizer(expression="\.", gaps=True)

text_schema = Schema(Text=TEXT(analyzer=text_analyzer, stored=True), 
                    Subject=TEXT(analyzer=text_analyzer, stored=True),
                    Newsgroup=TEXT(analyzer=newsgroup_analyzer, stored=True),
                    From=KEYWORD(stored=True),
                    Organization=KEYWORD(stored=True))


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_899515/507795809.py:2: SyntaxWarning: invalid escape sequence '\.'
  newsgroup_analyzer = RegexTokenizer(expression="\.", gaps=True)


In [ ]:
TextParser = QueryParser("Text", text_schema)
print(TextParser.parse(newsgroup_info_dicts[0]["Text"]))
AuthorParser = QueryParser("Author", text_schema)
print(AuthorParser.parse(newsgroup_info_dicts[0]["From"]))
OrganizationParser = QueryParser("Organization", text_schema)
print(OrganizationParser.parse(newsgroup_info_dicts[0]["Organization"]))
GroupParser = QueryParser("Newsgroup", text_schema)
print(GroupParser.parse(newsgroup_info_dicts[0]["Newsgroup"]))

(Text:wa AND Text:wonder AND Text:anyone AND Text:out AND Text:there AND Text:could AND Text:enlighten AND Text:me AND Text:car AND Text:saw AND Text:other AND Text:dai AND Text:door AND Text:sport AND Text:look AND Text:late AND Text:60 AND Text:earli AND Text:70 AND Text:call AND Text:bricklin AND Text:were AND Text:realli AND Text:small AND Text:addit AND Text:front AND Text:bumper AND Text:separ AND Text:rest AND Text:bodi AND Text:all AND Text:know AND Text:tellm AND Text:model AND Text:name AND Text:engin AND Text:spec AND Text:year AND Text:product AND Text:where AND Text:made AND Text:histori AND Text:whatev AND Text:info AND Text:funki AND Text:pleas AND Text:mail)
(Author:lerxst@wam.umd.edu AND Author:where's AND Author:my AND Author:thing)
(Organization:University AND Organization:of AND Organization:Maryland, AND Organization:College AND Organization:Park)
(Newsgroup:rec AND Newsgroup:autos)


#### Exercise 1.2 
rubric={accuracy:2}

Now, create the index for your posts and iterate over the `newsgroup_info_dicts` to add each post to your index. You should give each of the posts an`id` equal to its index in `newsgroup_info_dicts`.  The index creates an index in the specified directory.  Make sure to call writer.commit() after adding the documents to the index!  If you don't, and you lock the index, you can usually reset things by restartng your kernel.  Keep in mind that every time you call create_in, it will overwrite your old index. The newsgroup_info_dicts from the last part has ~11K entries, so you might want to create a status-tracker.

In [ ]:
import os.path
from whoosh.index import create_in
text_schema = Schema(Text=TEXT(analyzer=text_analyzer, stored=True), 
                    Subject=TEXT(analyzer=text_analyzer, stored=True),
                    Newsgroup=TEXT(analyzer=newsgroup_analyzer, stored=True),
                    From=KEYWORD(stored=True),
                    Organization=KEYWORD(stored=True),
                    id=NUMERIC(stored=True))
if not os.path.exists("index"):
    os.mkdir("index")
ix = create_in("index", text_schema)
writer = ix.writer()

In [ ]:
newsgroup_info_dicts[3]["Organization"]

'Harris Computer Systems Division'

In [ ]:
for i in range(len(newsgroup_info_dicts)):
    writer.add_document(
        id=i,
        From=newsgroup_info_dicts[i].get("From", ""),
        Subject=newsgroup_info_dicts[i].get("Subject", ""),
        Organization=newsgroup_info_dicts[i].get("Organization", ""),
        Text=newsgroup_info_dicts[i].get("Text", ""),
        Newsgroup=newsgroup_info_dicts[i].get("Newsgroup", ""),
    )
writer.commit()

#### Exercise 1.3 
rubric={accuracy:1}

Test your index by printing out all the authors of posts in one of the "comp" newsgroups (there are more than one) whose subject contains the word *floppy*. There should be 10 of them, and one of them is named *Stig*.

In [ ]:
query=QueryParser("",schema=text_schema).parse("Newsgroup:comp* AND Subject:floppy")

with ix.searcher() as searcher:
    results=searcher.search(query, limit=None) #The Results object acts like a list of the matched documents.
    print (results)
    for i in results:
        print(i["From"])

<Top 10 Results for And([Prefix('Newsgroup', 'comp'), Term('Subject', 'floppi')]) runtime=0.0015020550054032356>
limagen@hpwala.wal.hp.com
jdresser@altair.tymnet.com (Jay Dresser)
towwang@statler.engin.umich.edu (Tow Wang Hui)
venaas@flipper.pvv.unit.no (Stig Venaas)
balog@eniac.seas.upenn.edu (Eric J Balog)
tcking@uswnvg.com (Tim King)
jtrascap@nyx.cs.du.edu (Jim Trascapoulos)
bagels@gotham.East.Sun.COM (Alex Beigelman - NYC SE)
flyboy@spf.trw.com (Jeff Wright)
cctr132@csc.canterbury.ac.nz (Nick FitzGerald, PC Software Consultant, CSC, UoC, NZ)


### Exercise 2: Boolean IR

In this exercise, you will create your own boolean IR system and make sure it works the same way (or at least similarly) as Whoosh.

#### 2.1
rubric={accuracy:2}

First, create an inverted index for the 20 newsgroup corpus using a Python (default)dict. As you did above, use the index in newsgroup_info_dicts. In order to make the preprocessing consistent with Whoosh, you will need to use the analyzer you created in 1.1.


In [ ]:
newsgroup_info_dicts[0]

{'From': "lerxst@wam.umd.edu (where's my thing)",
 'Subject': 'WHAT car is this!?',
 'Nntp-Posting-Host': 'rac3.wam.umd.edu',
 'Organization': 'University of Maryland, College Park',
 'Lines': '15',
 'Text': 'I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 'Newsgroup': 'rec.autos'}

In [ ]:
inverted_index = defaultdict(set)
for i, post in enumerate(newsgroup_info_dicts):
    text_tokens = [t.text for t in text_analyzer(post["Text"])]
    tokens = text_tokens
    for t in tokens:
        inverted_index[t].add(i)

In [ ]:
assert "the" not in inverted_index
assert len(inverted_index["dog"]) == 70
assert 94 in inverted_index["dog"]
print("Success!")

Success!


#### 2.2
rubric={accuracy:3,efficiency:1}

Now you're going to create a boolean IR engine. You should complete the recursive `get_docs` function which takes your inverted index and a search "expression" and returns a set of all document ids which satisfy the expression. An expression is either a string (a single word), or a 2-tuple where the first element is a boolean operator ("or","and",or "not") and the second is either another tuple of expressions (if the operator is "or" or "and") or a single expression if the operator is "not" (HINT: for "not", a set consisting of all the document ids will be useful, and is provided below). For example, the following call to get_docs:

`get_docs(inverted_index,("and",("hit",("not", ("or", ("base","ball", "run")))))))`

Means that you want documents that contain the word *hit* and not any of the words *base*, *ball*, or *run*.  

The base case which gets the relevant documents for a single word using your inverted index is provided for you. You'll want to use set operators extensively here.

Recursion can be difficult to get your head around.  Try a test case for each logical operation - what happens when you have an "and"?  What about an "or"?  What about "not"?

In [ ]:
all_docs = set(range(len(newsgroup_info_dicts)))

def get_docs(inverted_index,expression):
    '''given an inverted index which provides a mapping from words to documents in a collection, evaluates expression according
    to boolean logic and returns a list of documents for which the expression holds
    '''
    if isinstance(expression, str):
        return inverted_index.get(expression, set())
    else:
        operator_type,operands = expression
        if operator_type == "not":
            return all_docs.difference(get_docs(inverted_index, operands))
            
        elif operator_type == "and":
            child_sets = [get_docs(inverted_index, op) for op in operands]
            return set.intersection(*child_sets)
        
        elif operator_type == "or":
            child_sets = [get_docs(inverted_index, op) for op in operands]
            return set.union(*child_sets)

#### 2.3
rubric={accuracy:1,quality:1}

Here, you will create tests which assert that the output of your boolean IR system is the same as Whoosh. You should have at least three tests:

1. Documents that contain the word *hit*
2. Documents that contain the words *hit*, *home*, and *run*
3. Documents that contain the word *hit* and not any of the words *base*, *ball*, or *run*. 

You should write a function that takes a Whoosh search and returns a set of document ids, the same output as `get_docs` (so it is easy to compare). Remember that you'll need to increase the default number of returned hits for Whoosh.

In [ ]:
def whoosh_docs(query):
    query=QueryParser(" ",schema=text_schema).parse(query)

    with ix.searcher() as searcher:
        results=searcher.search(query, limit=None)#The Results object acts like a list of the matched documents.
        return {r["id"] for r in results}

In [ ]:
wquery1 = "Text:hit"
wquery2 = "Text:hit AND Text:home AND Text:run"
wquery3 = "Text:hit AND NOT (Text:base OR Text:ball OR Text:run)"
bquery1 = "hit"
bquery2 = ("and", ("hit", "home", "run"))
bquery3 = ("and", ("hit", ("not", ("or", ("base", "ball", "run")))))

assert whoosh_docs(wquery1) == get_docs(inverted_index, bquery1)
assert whoosh_docs(wquery2) == get_docs(inverted_index, bquery2)
assert whoosh_docs(wquery3) == get_docs(inverted_index, bquery3)

### Exercise 3: Document relevance with Okapi BM25

In this exercise, you are again going to mimic the output of Whoosh by implementing [Okapi BM25 document relevance](https://en.wikipedia.org/wiki/Okapi_BM25), which is a special version of tf-idf that we will discuss in class (not the same used in sklearn!), as well as a simplified version of the vector-space model.

#### 3.1
rubric={accuracy:2,quality:1}

First you are going to calculate the *idf* part of the equation, as well as the average length of the texts (after preprocessing) which will be used in 3.2. First iterate over the corpus (`newsgroup_info_dicts`) and, after again using `simulate` for the Elasticsearch text analyzer, calculate an initial document frequency for each term, as well as the average length. Then, iterate over your df dict and create an corresponding idf dict. For Okapi BM25, idf is calculated as follows:

$$\text{IDF}(q_i) = \ln (\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5}+1)$$

where $q_i$ is the term (word type), $n(q_i)$ its document frequency (ie, how many documents it appears in), and $N$ the total number of documents. For quality, it's a good idea to have a function which calculates this idf.

In [ ]:
def calculate_idf(n, N):
    '''calculates Okapi BM25 idf based on the df n and the total number of
    documents N'''
    return log((1 + (N - n + 0.5)/(n+0.5)))

print(calculate_idf(2, 2))  ### Should be 0.1823215567939546

0.1823215567939546


In [ ]:
### Document frequency and avg length
N = len(newsgroup_info_dicts)
doc_freq = Counter()
avg_doc_len = defaultdict(int)
avg_len = 0
for i in range(N):
    tokens = [r.text for r in text_analyzer(newsgroup_info_dicts[i]["Text"])]
    avg_len += len(tokens)
    for t in set(tokens):
        doc_freq[t] += 1
avg_len /= N
### IDF
idf = {}
for term, nq in doc_freq.items():
    idf[term] = calculate_idf(nq, N)
avg_len

116.5888778550149

#### 3.2
rubric={accuracy:2,quality:1}

Now for the *tf* part, which will need to be calculated for each document. Iterate through the corpus again, and, for each document, first build a tf dictionary (a count of terms in the document), and then use that and the document frequencies you calculated in 3.1 to assign a tf-idf value for each term in each document. You are not using the raw term frequency, but a special term frequency calculated as follows:

$$\text{BM25-tf}(D,Q) = \frac{f(q_i, D)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgdl}})}$$

Where $q_i$ is the current term, f(q_i, D) is its frequency in the current document (the term frequency you have calculated), k_1 is the term saturation parameter with a (default) value of 1.2, b is the length normalization parameter with a (default) value of 0.75 (you can just use these values directly, you don't need to make them parameters here), $|D|$ is the token length of the current document, and $avg_dl$ is the average document length you calculated in 3.1. Then you should multiply it by the idf for that term you calculated in **3.1** to get a tf-idf score for that term in that document. When you have created a dictionary of all tf-idf scores in a document, append that dictionary to a list.

When calculating the tokens, you can use the `get_tokens` function you created earlier. This will ensure that you are still tokenizing and filtering in the same way as before. 

In [ ]:
def BM25_tf(tf, doc_length, avg_length):
    return tf/(tf + 1.2*(1 -0.75 + 0.75*(doc_length/avg_length)))

BM25_tf(2,9,9) ### Should be 0.625

0.625

In [ ]:
### Document frequency
N = len(newsgroup_info_dicts)
tf = defaultdict(Counter)
tf_idf = defaultdict(defaultdict)
for i in range(N):
    tokens = [r.text for r in text_analyzer(newsgroup_info_dicts[i]["Text"])]
    doc_length = len(tokens)
    tf[i].update(tokens)
    for t in tokens:
        bm_i = BM25_tf(tf[i][t], doc_length, avg_len)
        tf_idf[i][t] = bm_i * idf[t]

#### 3.3
rubric={accuracy:2,quality:1}

Now you are going to built a relevance ranking search engine. Create a function which takes a raw English query, the TFIDF corpus you just created, and the token analyzer. Then, iterate over the corpus and, for each term in the query, sum the corresponding tf-idf values from the document (if a term does not appear, its tf-idf defaults to 0) to get a relevance score for each document. Then rank all the documents in your corpus and return an ordered list of the top 10 ids, highest ranked first.  

In [ ]:
def BM25IR(query, analyzer, tfidfed_corpus):
    rankings = []
    relevance = 0
    for i in range(len(tfidfed_corpus)):
        tokens = [r.text for r in text_analyzer(query)]
        for t in tokens:
            # print(t)
            relevance += tfidfed_corpus[i].get(t, 0)
        rankings.append((i, relevance))
    rankings = sorted(rankings, key=lambda x: x[1], reverse=True)[:10]
    return list(map(lambda x: x[0], rankings))

BM25IR("Let's test this function!", text_analyzer, tf_idf)

[11068, 11069, 11070, 11071, 11072, 11073, 11074, 11075, 11076, 11045]

#### 3.4
rubric={accuracy:1,quality:1}

Create at least 3 example queries that are relevant to topic matters in the corpus and have at least 3 interesting (not stop) words. Print out the top document for each query as determined by your system, and show that your output is *almost* the same as the results of Whoosh using your index from Exercise 1 by comparing the two using P@10 (i.e., consider documents in the top 10 ranked results of Whoosh to be the "relevant" documents for this calculation); if you have done the rest of this problem correctly, you should get 1 or 0.9 for nearly all queries.

Be careful!  Your search engine should be using the text_analyzer you created above, but the Whoosh engine should take the result of the TextParser.  It's a fine distinction, but the Parser produces a generator, while the analyzer produces a list.

FYI, the reason your results may not be exactly the same as Whoosh is because Whoosh stores only approximations for some values. That is, your implementation of Okapi BM25 is in fact more accurate! You will note that it is also quite fast (and could be made even faster with the inverted index from Exercise 2), the main efficiency issue here is that you're using quite a lot of memory with all those Python dicts. Generally, this approach would not scale well to millions of documents.

In [ ]:
test_sents = ["job listings government", 
              "danger in my area",
              "university jobs internet"]

for sent in test_sents:
    retrieved = BM25IR(sent, text_analyzer, tf_idf)
    
    parsed_query = TextParser.parse(sent)
    
    with ix.searcher() as searcher:
        whoosh_results = searcher.search(parsed_query, limit=10)
        whoosh_ids = {r["id"] for r in whoosh_results}
    
    # calculate P@10


    print(f"{sent}, P@10: {len(whoosh_ids.difference(retrieved))/10}")
    

job listings government, P@10: 1.0
danger in my area, P@10: 1.0
university jobs internet, P@10: 0.7
